# AI Meeting Assistant: A Beginner-Friendly Walkthrough

**Goal:** Understand how a recording becomes a transcript, then becomes useful meeting notes.

You do not need an API key or an audio recording to read this notebook or run its default examples. Optional sections let you try real transcription and summarization when your environment is ready.

## Learning path

1. Understand the problem and the tools
2. Set up the project and find its files
3. Walk through a small meeting example
4. Understand Whisper transcription
5. Understand the prompt and LangChain workflow
6. Try optional live processing
7. Export results and launch the application
8. Check limitations, troubleshoot, and practice

**Prerequisites:** Basic Python variables, functions, and file paths. Allow about 20–30 minutes for the explanation; downloads and live processing take additional time.

## 1. What problem are we solving?

Imagine finishing a project meeting and asking: “What did we decide? Who will do the work? When is it due?” A recording contains the conversation, but finding those answers by listening again takes time.

This project uses two different AI tasks:

| Task | Input | Output | Tool |
| --- | --- | --- | --- |
| Transcription | Recorded speech | Written conversation | Whisper running on your computer |
| Summarization and extraction | Reviewed conversation | Organized meeting minutes | A text-generation model hosted on IBM watsonx.ai |

Gradio provides the screen with upload buttons and text boxes. LangChain connects a prompt template to the watsonx generation call. FFmpeg helps Whisper read audio formats. These tools have separate jobs: Whisper does not create the final action-item table.

### The complete journey

**Record/upload → Whisper → review transcript → prompt chain → watsonx → review notes → export.**

This app processes recordings after capture. It does not stream live minutes or recognize each speaker's identity.

## 2. Prepare your environment

Clone `RohanKurma/IBM_Works` and follow the project README to create and activate a Python 3.11 virtual environment. From the `watsonx-meeting-assistant` folder, install:

```bash
python -m pip install -r requirements.txt
python -m pip install notebook ipykernel
python -m ipykernel install --user --name meeting-assistant --display-name "Meeting Assistant"
python -m notebook
```

Open this notebook and choose the **Meeting Assistant** kernel. Installing a kernel makes Jupyter use the same Python environment as the project.

For real audio, install FFmpeg separately and confirm `ffmpeg -version` works. For IBM calls, copy `.env.example` to `.env` in the project folder and fill in your own values. Do not place secrets in notebook cells or saved outputs.

The next cell only locates the project. It works when Jupyter starts in this notebook's folder, the project folder, or the repository root.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents, cwd / "watsonx-meeting-assistant",
              cwd / "IBM_Works" / "watsonx-meeting-assistant"]
PROJECT_DIR = next((p for p in candidates if (p / "meeting_assistant.py").is_file()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("Open the notebook from inside the cloned IBM_Works project.")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from meeting_assistant import validate_transcript, transcribe, generate_notes, export_results
print("Project found:", PROJECT_DIR.name)

### How the files fit together

| File | Think of it as… | Main responsibility |
| --- | --- | --- |
| `meeting_assistant.py` | The engine | Transcribe audio, validate text, request notes, write files |
| `app.py` | The dashboard | Connect Gradio controls to the engine |
| `cli.py` | The terminal entry point | Run the workflow without opening the web app |
| `.env` | Your local configuration | Store credentials and model choice outside source code |
| `tests/` | Automated checks | Check expected behavior without calling paid services |

The notebook imports the existing engine rather than maintaining a second implementation. Changes to the engine are therefore reflected in the examples.

## 3. Start with a tiny meeting

We can learn the workflow using text before introducing audio. The following conversation is **invented for this tutorial**. Notice the difference between a decision, an assigned task, and a suggestion.

In [ ]:
sample_transcript = """Maya: We agreed to pilot the dashboard with five teachers.
Rohan: I will prepare the demo by Friday.
Maya: I will collect feedback, but we have not set a deadline.
Rohan: We could add a mobile app later; that is only a suggestion.
Maya: We still need to decide how to measure the pilot's success."""

reviewed_transcript = validate_transcript(sample_transcript)
print(reviewed_transcript)

### What good notes should capture

The next example is **manually written**, not a response from IBM. It is a reference to help you evaluate an actual model result. The system should preserve “Friday” as written because the meeting date is unknown. It should not turn a suggested mobile app into an approved decision.

In [ ]:
reference_notes = """# Meeting notes

## Summary
The team agreed to pilot a dashboard with five teachers and discussed preparation and feedback.

## Key discussion points
- Prepare a demo and collect pilot feedback.
- A mobile app was suggested as a possible future addition.

## Decisions
- Pilot the dashboard with five teachers.

## Action items
| Task | Owner | Deadline |
| --- | --- | --- |
| Prepare the demo | Rohan | Friday |
| Collect feedback | Maya | Not specified |

## Open questions
- How should the pilot's success be measured?
"""
from IPython.display import display, Markdown
display(Markdown(reference_notes))

## 4. How Whisper turns audio into words

`transcribe(audio_path, model_name)` in the engine:

1. Checks that the input file exists and the model choice is supported.
2. Loads a local Whisper model. The first call may download model weights.
3. Asks Whisper to decode and transcribe the audio with `fp16=False`, which supports CPU execution.
4. Removes surrounding whitespace and reports an error if no speech was found.

The model loader is cached, so repeated calls with the same model can reuse it in the same Python process. The app offers `tiny`, `base`, and `small`; larger models generally require more resources, but quality must be evaluated on your recordings.

Whisper can mishear names, numbers, accents, or overlapping speech. Review the transcript **before** asking the LLM to summarize it. The project does not perform automatic speaker labeling.

The following optional cell is off by default. To try it, set a real file path and change the flag. Processing happens on your machine; downloading model weights still requires internet access.

In [ ]:
RUN_AUDIO = False
AUDIO_PATH = PROJECT_DIR / "my_meeting.wav"  # Replace with your recording.
audio_transcript = None

if RUN_AUDIO:
    audio_transcript = transcribe(AUDIO_PATH, model_name="base")
    print(audio_transcript)
else:
    print("Audio processing skipped. The tutorial continues with the sample transcript.")

## 5. How the transcript becomes structured notes

A language model needs instructions about the output we want. The project's prompt asks it to use these sections:

- Summary
- Key discussion points
- Decisions
- Action items, with Task / Owner / Deadline columns
- Open questions

It also asks the model to use only transcript evidence, preserve uncertainty, and write “Not specified” for missing information. These instructions help guide the result; they do not guarantee correctness.

### What do PromptTemplate and a chain do?

`PromptTemplate` is a reusable instruction with a `{transcript}` placeholder. The chain fills that placeholder and passes the finished prompt to the IBM SDK.

The implementation uses this pattern:

```python
chain = prompt | RunnableLambda(
    lambda value: model.generate_text(prompt=value.to_string())
)
notes = chain.invoke({"transcript": transcript})
```

Read `|` as “send the result to the next step.” `RunnableLambda` wraps the Python function that calls watsonx. This workflow uses one generation call rather than an autonomous agent or a retrieval system.

The example below demonstrates the idea of filling a placeholder using ordinary Python. It does not send data to any service.

In [ ]:
teaching_template = "Summarize only the following meeting: <transcript>{transcript}</transcript>"
teaching_prompt = teaching_template.format(transcript=reviewed_transcript)
print(teaching_prompt)
print("\nThis short teaching prompt is not the full production prompt.")

### Why validate the input first?

The engine rejects empty transcripts and transcripts longer than 24,000 characters. This prevents silent truncation and catches obvious input mistakes. Characters are not tokens: a specific IBM model may still require a shorter input.

Long meetings need manual splitting in this version. Splitting can lose context across segments, so review the combined notes carefully. The following cell demonstrates validation without making an API request.

In [ ]:
for label, text in [("Empty transcript", "   "), ("Oversized transcript", "x" * 24001)]:
    try:
        validate_transcript(text)
    except ValueError as error:
        print(f"{label}: {error}")

## 6. Optional: generate real notes with watsonx

Configure the following in the project `.env` file:

| Variable | Meaning |
| --- | --- |
| `WATSONX_APIKEY` | Your IBM Cloud API key |
| `WATSONX_PROJECT_ID` | Your watsonx project identifier |
| `WATSONX_URL` | The endpoint for your IBM region |
| `WATSONX_MODEL_ID` | A text-generation model available to your account and region |

Do not assume a model from a hosted lab is available in your own account. IBM inference may incur charges.

The next cell checks only whether settings exist; it never prints their values. It is off by default. Setting `RUN_IBM=True` sends `reviewed_transcript` to IBM. If you replace the sample with real meeting text, review what you are sending first.

In [ ]:
RUN_IBM = False
live_notes = None

if RUN_IBM:
    import os
    from dotenv import load_dotenv
    load_dotenv(PROJECT_DIR / ".env")
    keys = ["WATSONX_APIKEY", "WATSONX_PROJECT_ID", "WATSONX_MODEL_ID"]
    missing = [key for key in keys if not os.getenv(key, "").strip()]
    if missing:
        raise ValueError("Set configuration variables: " + ", ".join(missing))
    live_notes = generate_notes(reviewed_transcript)
    display(Markdown(live_notes))
else:
    print("IBM request skipped. No transcript was sent to IBM by this cell.")

### Review the model result

Compare generated notes against the transcript:

| Check | Question to ask |
| --- | --- |
| Evidence | Can I point to the sentence supporting each claim? |
| Decisions | Did the model confuse a suggestion with an agreement? |
| Ownership | Were task owners actually named? |
| Dates | Were deadlines preserved without guessing a calendar date? |
| Completeness | Were important decisions or tasks omitted? |
| Uncertainty | Did the output retain unresolved questions? |

A well-formatted response can still contain mistakes. The application does not automatically verify facts or guarantee that every required section appears.

## 7. Export the transcript and notes

`export_results()` writes UTF-8 text and Markdown files with a unique shared identifier. A new export does not overwrite a previous meeting.

The next cell uses a temporary folder that is removed automatically after the demonstration. It exports the manually written reference unless you ran the optional IBM cell. To keep files yourself, call `export_results(reviewed_transcript, notes_to_export, PROJECT_DIR / "outputs")`.

In [ ]:
import tempfile

notes_to_export = live_notes if live_notes else reference_notes
print("Exporting:", "live IBM notes" if live_notes else "manually written tutorial notes")
with tempfile.TemporaryDirectory(prefix="meeting-tutorial-") as folder:
    transcript_file, notes_file = export_results(reviewed_transcript, notes_to_export, folder)
    print("Transcript file:", Path(transcript_file).name)
    print("Notes file:", Path(notes_file).name)
    assert Path(transcript_file).read_text(encoding="utf-8").strip() == reviewed_transcript
    assert Path(notes_file).read_text(encoding="utf-8").strip() == notes_to_export.strip()
print("Export checked; the temporary example files have been removed.")

## 8. Use the complete app

Run these commands in a terminal from the project folder, with your virtual environment activated.

### Browser interface

```bash
python app.py
```

Open `http://127.0.0.1:7860`. Upload or record audio, click **Transcribe audio**, review the text, then click **Generate notes**. You can also paste a transcript to skip transcription. Download the resulting files using the file controls.

### Command-line interface

```bash
python cli.py --audio meeting.wav
python cli.py --audio meeting.wav --model tiny --transcribe-only
python cli.py --transcript transcript.txt --output-dir outputs/team-meeting
```

The CLI saves the transcript before requesting notes. If IBM is unavailable, the saved transcript remains. Transcription-only mode also creates a companion notes file with a placeholder.

### Automated checks

```bash
python -m unittest discover -s tests -v
```

These are offline tests of validation, transcription wiring and file export behavior. They do not measure speech-recognition accuracy or prove the IBM service is configured correctly.

## 9. Troubleshooting and responsible use

| Symptom | What to check |
| --- | --- |
| Python cannot import a package | Select the project kernel and install requirements in that environment |
| Audio cannot be decoded | Confirm FFmpeg works and the file can be played |
| First transcription is slow | Whisper may be downloading weights or running on CPU |
| No speech found | Try a shorter, clearer recording with audible speech |
| IBM configuration error | Check the required `.env` names, regional URL, permissions and model access |
| Context limit error | Shorten the transcript; 24,000 characters is an app limit, not a model guarantee |
| Notes contain invented details | Compare with the transcript and correct them before sharing |
| Notebook cannot find project | Open it within the cloned repository; retain the adjacent project files |

Audio is processed locally by Whisper. Notes generation sends the transcript to IBM. Model downloads and service calls need network access. Gradio uploads, downloads and application exports can remain in temporary/local storage; remove sensitive files when no longer needed. The tutorial's temporary export example cleans up its own files.

The app has no production authentication, diarization, automatic long-meeting chunking, or automatic accuracy evaluation. Keep the Gradio demo local. A prompt telling the model to ignore transcript instructions is guidance, not a complete prompt-injection defense.

## 10. Practice: check your understanding

1. Why should the transcript be reviewed before generating notes?
2. If someone says “We could launch next month,” should that become a decision?
3. Can transcription work without an IBM key? Which stage needs the key?
4. Why should a missing deadline stay “Not specified”?
5. What is the difference between a character limit and a model's token limit?

### Suggested answers

1. Transcription errors can propagate into the summary and assigned tasks.
2. No. It is a proposal unless the conversation establishes agreement.
3. Yes, local Whisper transcription can run without an IBM key. watsonx generation requires configured credentials.
4. Invented deadlines could mislead the team.
5. Characters measure text length; models use tokens, whose relationship to characters varies.

### Small experiment

Copy `sample_transcript`, add an explicit owner and deadline for a new task, and edit the reference action-item table by hand. If you have IBM access, generate notes for both versions and compare whether only evidence-supported details changed.

### Explain this project in one minute

“This meeting assistant uses Whisper on my computer to turn recorded speech into text. I review that text, then a LangChain prompt sends it to a watsonx model to organize the discussion into summaries, decisions and tasks. Gradio makes the workflow easy to use, while a CLI supports batch runs. The project exports editable files and checks inputs, but people still review the final notes.”

---

**Attribution:** This walkthrough explains the implementation in `IBM_Works/watsonx-meeting-assistant`, built from the outline in the supplied Skills Network meeting-assistant document. See the project's `NOTICE.md` for source limitations. The sample meeting and reference notes in this notebook are fictional teaching examples.